In [ ]:
import pandas as pd
import numpy as np

file_path = 'docs/YCDL.xlsx'
sheets = ['HBa1C', 'LDL', 'HDL', 'Triglycerid']

dfs = []
col_mapping = {
    'MaICD Nội Trú': 'MaICD Bn nội trú',
    'MaICD Nội trú': 'MaICD Bn nội trú',
    'MaICD Ngoại Trú': 'MaICD BN ngoại trú',
    'MaICD Ngoại trú': 'MaICD BN ngoại trú',
    # Giả định file excel có cột ngày. Hãy đổi tên nếu file gốc của bạn khác
    'NgayThucHien': 'NgayThucHien' 
}

# 1. Định nghĩa các cột cơ bản
base_cols = ['TenBenhNhan', 'SoVaoVien', 'NamSinh', 'GioiTinh', 'NgayThucHien', 'MaICD Bn nội trú', 'MaICD BN ngoại trú']

for sheet in sheets:
    df = pd.read_excel(file_path, sheet_name=sheet)
    df = df.rename(columns=col_mapping)
    
    # Fill NaN cho các cột cơ bản nếu thiếu
    for col in base_cols:
        if col not in df.columns:
            df[col] = np.nan
            
    if 'KetQua' in df.columns:
        # Ép kiểu dữ liệu về số (float)
        df[sheet] = pd.to_numeric(df['KetQua'], errors='coerce')
    else:
        df[sheet] = np.nan
        
    cols_to_keep = base_cols + [sheet]
    df = df[cols_to_keep]
    dfs.append(df)

# 2. Nối dữ liệu và Sắp xếp
df_concat = pd.concat(dfs, ignore_index=True)
df_concat = df_concat.sort_values(by=['TenBenhNhan', 'SoVaoVien', 'NgayThucHien'])

# 3. Khai báo các hàm tổng hợp (Feature Aggregation)
group_cols = ['TenBenhNhan', 'SoVaoVien', 'NamSinh', 'GioiTinh']
numeric_agg = ['mean', 'max', 'min', 'std', 'last']

def join_unique_icd(x):
    valid_icds = x.dropna().astype(str).str.strip().unique()
    return ', '.join(valid_icds) if len(valid_icds) > 0 else None

agg_funcs = {
    'MaICD Bn nội trú': join_unique_icd,
    'MaICD BN ngoại trú': join_unique_icd,
    'HBa1C': numeric_agg,
    'LDL': numeric_agg,
    'HDL': numeric_agg,
    'Triglycerid': numeric_agg
}

df_grouped = df_concat.groupby(group_cols, dropna=False).agg(agg_funcs)

# 4. Làm phẳng cấu trúc cột MultiIndex
new_columns = []
for col in df_grouped.columns:
    col_name = col[0]
    func_name = col[1]
    
    if func_name in numeric_agg:
        new_columns.append(f"{col_name}_{func_name}")
    else:
        new_columns.append(col_name)

df_grouped.columns = new_columns
df_final = df_grouped.reset_index()

# 5. XỬ LÝ LỖI LOGIC NÂNG CAO: Chuẩn hóa cột _std
# Chỉ điền 0 cho std khi bệnh nhân có xét nghiệm (mean != NaN) nhưng chỉ đo 1 lần (std == NaN)
for sheet in sheets:
    mean_col = f"{sheet}_mean"
    std_col = f"{sheet}_std"
    
    if mean_col in df_final.columns and std_col in df_final.columns:
        # mask = Bệnh nhân CÓ thực hiện xét nghiệm VÀ giá trị std bị rỗng
        mask = df_final[mean_col].notnull() & df_final[std_col].isnull()
        df_final.loc[mask, std_col] = 0.0

# 6. Hoàn thiện và Xuất file
df_final = df_final.sort_values(by=['TenBenhNhan']).reset_index(drop=True)


# Create target for data
target_codes = ['I21', 'I22', 'I64', 'I50']

def create_target(row):
    # Nối tất cả mã ICD thành một chuỗi duy nhất để kiểm tra
    full_icd = str(row['MaICD Bn nội trú']) + " " + str(row['MaICD BN ngoại trú'])
    if any(code in full_icd for code in target_codes):
        return 1 # Có biến chứng
    return 0 # Không có biến chứng

df_final['Target'] = df_final.apply(create_target, axis=1)

# Chuyển đổi toàn bộ NaN còn lại (các bệnh nhân không xét nghiệm) thành None để xuất ra Excel
df_final = df_final.replace({np.nan: None})

output_path = 'docs/YCDL_Total.xlsx'
with pd.ExcelWriter(output_path) as writer:
    df_final.to_excel(writer, sheet_name='Total', index=False)

print('Data successfully exported to', output_path)

Data successfully exported to docs/YCDL_Total.xlsx


In [11]:
import pandas as pd
import numpy as np
import re

file_path = 'docs/YCDL.xlsx'
sheets = ['HBa1C', 'LDL', 'HDL', 'Triglycerid']

dfs = []
col_mapping = {
    'MaICD Nội Trú': 'MaICD Bn nội trú',
    'MaICD Nội trú': 'MaICD Bn nội trú',
    'MaICD Ngoại Trú': 'MaICD BN ngoại trú',
    'MaICD Ngoại trú': 'MaICD BN ngoại trú',
    # Bạn cần đảm bảo file Excel có cột thời gian để tính khoảng thời gian 12 tháng
    'NgayThucHien': 'NgayThucHien' 
}

base_cols = ['TenBenhNhan', 'SoVaoVien', 'NamSinh', 'GioiTinh', 'NgayThucHien', 'MaICD Bn nội trú', 'MaICD BN ngoại trú']

# ---------------------------------------------------------
# BƯỚC 1: ĐỌC VÀ CHUẨN HÓA DỮ LIỆU THÔ
# ---------------------------------------------------------
for sheet in sheets:
    df = pd.read_excel(file_path, sheet_name=sheet)
    df = df.rename(columns=col_mapping)
    
    for col in base_cols:
        if col not in df.columns:
            df[col] = np.nan
            
    # Chuyển đổi Ngày xét nghiệm sang định dạng datetime để tính toán timeline
    if 'NgayThucHien' in df.columns:
        df['NgayThucHien'] = pd.to_datetime(df['NgayThucHien'], errors='coerce')

    if 'KetQua' in df.columns:
        df[sheet] = pd.to_numeric(df['KetQua'], errors='coerce')
    else:
        df[sheet] = np.nan
        
    cols_to_keep = base_cols + [sheet]
    df = df[cols_to_keep]
    dfs.append(df)

df_concat = pd.concat(dfs, ignore_index=True)
df_concat = df_concat.sort_values(by=['TenBenhNhan', 'SoVaoVien', 'NgayThucHien'])

# ---------------------------------------------------------
# BƯỚC 2: GOM CỤM DỮ LIỆU (FEATURE AGGREGATION)
# ---------------------------------------------------------
group_cols = ['TenBenhNhan', 'SoVaoVien', 'NamSinh', 'GioiTinh']
numeric_agg = ['mean', 'max', 'min', 'std', 'last']

def join_unique_icd(x):
    valid_icds = x.dropna().astype(str).str.strip().unique()
    return ', '.join(valid_icds) if len(valid_icds) > 0 else None

agg_funcs = {
    'MaICD Bn nội trú': join_unique_icd,
    'MaICD BN ngoại trú': join_unique_icd,
    # Lấy min/max ngày xét nghiệm để kiểm tra bệnh nhân có đủ 12 tháng theo dõi không
    'NgayThucHien': ['min', 'max'], 
    'HBa1C': numeric_agg,
    'LDL': numeric_agg,
    'HDL': numeric_agg,
    'Triglycerid': numeric_agg
}

df_grouped = df_concat.groupby(group_cols, dropna=False).agg(agg_funcs)

# Làm phẳng cột
new_columns = []
for col in df_grouped.columns:
    col_name = col[0]
    func_name = col[1]
    
    if func_name in numeric_agg + ['min', 'max']:
        new_columns.append(f"{col_name}_{func_name}")
    else:
        new_columns.append(col_name)

df_grouped.columns = new_columns
df_final = df_grouped.reset_index()

# ---------------------------------------------------------
# BƯỚC 3: BỘ LỌC DỮ LIỆU THEO TIÊU CHÍ ĐỐI TƯỢNG ĐỒ ÁN
# ---------------------------------------------------------

# 3.1. Lọc theo Tuổi (Age >= 18) 
# Tính theo mốc năm 2026 của đồ án 
df_final['Age'] = 2026 - pd.to_numeric(df_final['NamSinh'], errors='coerce')
df_final = df_final[df_final['Age'] >= 18]

# 3.2. Lọc thời gian theo dõi >= 12 tháng (365 ngày) 
df_final['TheoDoi_Ngay'] = (df_final['NgayThucHien_max'] - df_final['NgayThucHien_min']).dt.days
df_final = df_final[df_final['TheoDoi_Ngay'] >= 365]

# 3.3. Lọc chẩn đoán (I10-I15) và loại trừ bệnh ác tính (C) 
def filter_patients(row):
    full_icd = str(row['MaICD Bn nội trú']) + " " + str(row['MaICD BN ngoại trú'])
    has_hypertension = bool(re.search(r'I1[0-5]', full_icd))
    has_malignant = bool(re.search(r'C\d{2}', full_icd)) # Bắt mã C theo sau là số (vd C00, C97)
    return has_hypertension and not has_malignant

df_final = df_final[df_final.apply(filter_patients, axis=1)]

# 3.4. Lọc dữ liệu thiếu (Giữ lại nếu có ít nhất 1 loại xét nghiệm) 
critical_features = ['HBa1C_mean', 'LDL_mean', 'HDL_mean', 'Triglycerid_mean']
df_final = df_final.dropna(subset=critical_features, thresh=1)

# ---------------------------------------------------------
# BƯỚC 4: TIỀN XỬ LÝ NÂNG CAO & TẠO TARGET
# ---------------------------------------------------------

# 4.1. Xử lý logic std NaN (Chỉ gán 0 khi có đo nhưng đo 1 lần)
for sheet in sheets:
    mean_col = f"{sheet}_mean"
    std_col = f"{sheet}_std"
    if mean_col in df_final.columns and std_col in df_final.columns:
        mask = df_final[mean_col].notnull() & df_final[std_col].isnull()
        df_final.loc[mask, std_col] = 0.0

# 4.2. Tạo Target Label (Có biến chứng = 1, Không = 0)
target_codes = ['I21', 'I22', 'I64', 'I50']
def create_target(row):
    full_icd = str(row['MaICD Bn nội trú']) + " " + str(row['MaICD BN ngoại trú'])
    if any(code in full_icd for code in target_codes):
        return 1
    return 0

df_final['Target'] = df_final.apply(create_target, axis=1)

# ---------------------------------------------------------
# BƯỚC 5: DỌN DẸP VÀ XUẤT FILE
# ---------------------------------------------------------
# Bỏ các cột tính toán trung gian
cols_to_drop = ['NgayThucHien_min', 'NgayThucHien_max', 'TheoDoi_Ngay']
df_final = df_final.drop(columns=cols_to_drop, errors='ignore')

# Sắp xếp và làm sạch NaN
df_final = df_final.sort_values(by=['TenBenhNhan']).reset_index(drop=True)
df_final = df_final.replace({np.nan: None})

output_path = 'docs/YCDL_Total-temp.xlsx'
with pd.ExcelWriter(output_path) as writer:
    df_final.to_excel(writer, sheet_name='Total', index=False)

print(f'Pipeline completed! Data exported to {output_path}. Total valid patients remaining: {len(df_final)}')

Pipeline completed! Data exported to docs/YCDL_Total-temp.xlsx. Total valid patients remaining: 12355


In [13]:
import pandas as pd
import numpy as np
import re

# Cấu hình đường dẫn và danh sách các xét nghiệm cần trích xuất
file_path = 'docs/YCDL.xlsx'
sheets = ['HBa1C', 'LDL', 'HDL', 'Triglycerid']

# Ánh xạ tên cột để đồng nhất dữ liệu từ các sheet khác nhau
col_mapping = {
    'MaICD Nội Trú': 'MaICD Bn nội trú',
    'MaICD Nội trú': 'MaICD Bn nội trú',
    'MaICD Ngoại Trú': 'MaICD BN ngoại trú',
    'MaICD Ngoại trú': 'MaICD BN ngoại trú',
    'NgayThucHien': 'NgayThucHien' 
}

# Các cột thông tin cơ bản cần thiết cho mỗi bản ghi
base_cols = ['TenBenhNhan', 'SoVaoVien', 'NamSinh', 'GioiTinh', 'NgayThucHien', 'MaICD Bn nội trú', 'MaICD BN ngoại trú']

# ---------------------------------------------------------
# BƯỚC 1: ĐỌC DỮ LIỆU VÀ LỌC THEO KHUNG THỜI GIAN ĐỀ CƯƠNG
# ---------------------------------------------------------
dfs = []
for sheet in sheets:
    df = pd.read_excel(file_path, sheet_name=sheet)
    df = df.rename(columns=col_mapping)
    
    # Đảm bảo có đủ các cột cơ bản
    for col in base_cols:
        if col not in df.columns:
            df[col] = np.nan
            
    # Ép kiểu dữ liệu ngày tháng và kết quả xét nghiệm
    if 'NgayThucHien' in df.columns:
        df['NgayThucHien'] = pd.to_datetime(df['NgayThucHien'], errors='coerce')

    if 'KetQua' in df.columns:
        df[sheet] = pd.to_numeric(df['KetQua'], errors='coerce')
    else:
        df[sheet] = np.nan
        
    dfs.append(df[base_cols + [sheet]])

# Nối tất cả dữ liệu thô
df_concat = pd.concat(dfs, ignore_index=True)

# LỌC THEO TIÊU CHÍ THỜI GIAN KHÁM/NHẬP VIỆN (01/01/2023 - 28/02/2026)
# 
start_date = pd.to_datetime('2023-01-01')
end_date = pd.to_datetime('2026-02-28')

df_concat = df_concat.dropna(subset=['NgayThucHien'])
df_concat = df_concat[(df_concat['NgayThucHien'] >= start_date) & (df_concat['NgayThucHien'] <= end_date)]

# Sắp xếp dữ liệu theo bệnh nhân và thời gian
df_concat = df_concat.sort_values(by=['TenBenhNhan', 'SoVaoVien', 'NgayThucHien'])

# ---------------------------------------------------------
# BƯỚC 2: GOM CỤM DỮ LIỆU (AGGREGATION)
# ---------------------------------------------------------
group_cols = ['TenBenhNhan', 'SoVaoVien', 'NamSinh', 'GioiTinh']
numeric_agg = ['mean', 'max', 'min', 'std', 'last'] # [cite: 50]

def join_unique_icd(x):
    valid_icds = x.dropna().astype(str).str.strip().unique()
    return ', '.join(valid_icds) if len(valid_icds) > 0 else None

agg_funcs = {
    'MaICD Bn nội trú': join_unique_icd,
    'MaICD BN ngoại trú': join_unique_icd,
    'NgayThucHien': ['min', 'max'], # Để tính thời gian theo dõi
    'HBa1C': numeric_agg,
    'LDL': numeric_agg,
    'HDL': numeric_agg,
    'Triglycerid': numeric_agg
}

df_grouped = df_concat.groupby(group_cols, dropna=False).agg(agg_funcs)

# Làm phẳng tiêu đề cột (Flatten MultiIndex)
df_grouped.columns = [f"{c[0]}_{c[1]}" if c[1] in numeric_agg + ['min', 'max'] else c[0] for c in df_grouped.columns]
df_final = df_grouped.reset_index()

# ---------------------------------------------------------
# BƯỚC 3: BỘ LỌC ĐỐI TƯỢNG (INCLUSION/EXCLUSION CRITERIA)
# ---------------------------------------------------------

# 3.1. Lọc theo độ tuổi (Tuổi >= 18 tại mốc 2026) 
df_final['Age'] = 2026 - pd.to_numeric(df_final['NamSinh'], errors='coerce')
df_final = df_final[df_final['Age'] >= 18]

# 3.2. Lọc thời gian theo dõi tối thiểu 12 tháng 
df_final['FollowUp_Days'] = (df_final['NgayThucHien_max'] - df_final['NgayThucHien_min']).dt.days
df_final = df_final[df_final['FollowUp_Days'] >= 365]

# 3.3. Lọc chẩn đoán Tăng huyết áp (I10-I15) và loại trừ bệnh ác tính (C) 
def validate_diagnosis(row):
    full_icd = str(row['MaICD Bn nội trú']) + " " + str(row['MaICD BN ngoại trú'])
    is_htn = bool(re.search(r'I1[0-5]', full_icd)) # Chẩn đoán I10-I15
    is_malignant = bool(re.search(r'C\d{2}', full_icd)) # Loại trừ mã C
    return is_htn and not is_malignant

df_final = df_final[df_final.apply(validate_diagnosis, axis=1)]

# ---------------------------------------------------------
# BƯỚC 4: TIỀN XỬ LÝ VÀ TẠO NHÃN MỤC TIÊU (TARGET)
# ---------------------------------------------------------

# 4.1. Xử lý logic std NaN (Gán bằng 0 nếu chỉ đo 1 lần)
for s in sheets:
    mean_col, std_col = f"{s}_mean", f"{s}_std"
    if mean_col in df_final.columns and std_col in df_final.columns:
        mask = df_final[mean_col].notnull() & df_final[std_col].isnull()
        df_final.loc[mask, std_col] = 0.0

# 4.2. Tạo nhãn Target dựa trên biến chứng tim mạch 
target_codes = ['I21', 'I22', 'I64', 'I50'] # NMCT, Đột quỵ, Suy tim
def define_target(row):
    full_icd = str(row['MaICD Bn nội trú']) + " " + str(row['MaICD BN ngoại trú'])
    return 1 if any(code in full_icd for code in target_codes) else 0

df_final['Target'] = df_final.apply(define_target, axis=1)

# ---------------------------------------------------------
# BƯỚC 5: XUẤT DỮ LIỆU
# ---------------------------------------------------------
# Loại bỏ các cột phụ dùng để lọc
df_final = df_final.drop(columns=['NgayThucHien_min', 'NgayThucHien_max', 'FollowUp_Days'], errors='ignore')
df_final = df_final.sort_values(by=['TenBenhNhan']).replace({np.nan: None})

output_path = 'docs/YCDL_Total-temp2.xlsx'
with pd.ExcelWriter(output_path) as writer:
    df_final.to_excel(writer, sheet_name='Total', index=False)

print(f"Xử lý hoàn tất! Tổng số bệnh nhân hợp lệ: {len(df_final)}")

Xử lý hoàn tất! Tổng số bệnh nhân hợp lệ: 12355


In [ ]:
import pandas as pd
import numpy as np
import re

# ─────────────────────────────────────────────────────────────────
# STEP 1: Load source file
# ─────────────────────────────────────────────────────────────────
source_path = 'docs/YCDL_Total-temp2.xlsx'
df = pd.read_excel(source_path, sheet_name='Total')

# ─────────────────────────────────────────────────────────────────
# STEP 2: Create a temporary helper column with all ICD strings merged
# ─────────────────────────────────────────────────────────────────
df['_all_icd'] = (
    df['MaICD Bn nội trú'].fillna('').astype(str)
    + ' '
    + df['MaICD BN ngoại trú'].fillna('').astype(str)
)

# ─────────────────────────────────────────────────────────────────
# STEP 3: Define ICD-10 mapping rules
#   - Use word boundaries / non-digit lookaheads to prevent partial matches
#     e.g., I50 must NOT match I500
# ─────────────────────────────────────────────────────────────────
icd_feature_map = {
    'dai_thao_duong': r'\b(E10|E11|E12|E13|E14)(\.[0-9]+)?\b',   # Diabetes Mellitus
    'rl_lipid_mau':   r'\bE78(\.[0-9]+)?\b',                      # Lipoprotein metabolism disorders
    'suy_than_man':   r'\bN18(\.[0-9]+)?\b',                      # Chronic kidney disease
    'benh_mach_vanh': r'\bI25\.0\b',                              # Atherosclerotic cardiovascular disease (exact)
    'suy_tim':        r'\bI50(\.[0-9]+)?\b',                      # Heart failure
    'dot_quy':        r'\bI64\b',                                  # Stroke, not specified
    'rung_nhi':       r'\bI48(\.[0-9]+)?\b',                      # Atrial fibrillation and flutter
    'Tang_huyet_ap':  r'\b(I10|I11|I12|I13|I14|I15)(\.[0-9]+)?\b',# Hypertension
}

# ─────────────────────────────────────────────────────────────────
# STEP 4: Apply each pattern, cast to int64, fill missing as 0
# ─────────────────────────────────────────────────────────────────
for feature_name, pattern in icd_feature_map.items():
    df[feature_name] = (
        df['_all_icd']
        .str.contains(pattern, flags=re.IGNORECASE, regex=True, na=False)
        .astype('int64')
    )

# ─────────────────────────────────────────────────────────────────
# STEP 5: Drop helper column and save output
# ─────────────────────────────────────────────────────────────────
df.drop(columns=['_all_icd'], inplace=True)

output_path = 'docs/YCDL_Features_Mapped.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Total', index=False)

print(f"Feature engineering complete. Output saved to: {output_path}")
print(f"Shape: {df.shape}")
print("\nNew feature columns distribution:")
for feat in icd_feature_map:
    count = df[feat].sum()
    pct   = df[feat].mean() * 100
    print(f"  {feat:<20s}: {count:>4d} patients ({pct:5.1f}%)  | dtype: {df[feat].dtype}")

